In [1]:
import numpy as np
import base64
import struct
from cobs import cobs
from collections import namedtuple
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.pyplot import plot
from tqdm.notebook import tqdm
import scipy.signal
from scipy.io import savemat
# import imageio
import pandas as pd
import scipy.io as spio
import os

In [2]:
# Format package
DataPacketDesc = {'type': 'B',
				  'size': 'B',
				  'crc16': 'H',
				  'packetID': 'I',
				  'us_start': 'I',
				  'us_end': 'I',
				  'analog': '8H',
				  'states': '8l',
				  'digitalIn': '2H',
				  'digitalOut': '3B',
				  'padding': 'x'}


DataPacket = namedtuple('DataPacket', DataPacketDesc.keys())
DataPacketStruct = '<' + ''.join(DataPacketDesc.values())
DataPacketSize = struct.calcsize(DataPacketStruct)

# package with non-digital data
dtype_no_digital = [
	('type', np.uint8),
	('size', np.uint8),
	('crc16', np.uint16),
	('packetID', np.uint32),
	('us_start', np.uint32),
	('us_end', np.uint32),
	('analog', np.uint16, (8, )),
	('states', np.uint32, (8, ))]

# DigitalIn and DigitalOut
dtype_w_digital = dtype_no_digital + [('digital_in', np.uint16, (16, )), ('digital_out', np.uint16, (16, ))]

# Creating array with all the data (differentiation digital/non digital)
np_DataPacketType_noDigital = np.dtype(dtype_no_digital)
np_DataPacketType_withDigital = np.dtype(dtype_w_digital)

In [3]:
# function to count the packet number
def count_lines(fp):
	def _make_gen(reader):
		b = reader(2**17)
		while b:
			yield b
			b = reader(2**17)
	with open(fp, 'rb') as f:
		count = sum(buf.count(b'\n') for buf in _make_gen(f.raw.read))
	return count

def unpack_data_packet(dp):
	s = struct.unpack(DataPacketStruct, dp)
	up = DataPacket(type=s[0], size=s[1], crc16=s[2], packetID=s[3], us_start=s[4], us_end=s[5],
						analog=s[6:14], states=s[14:22], digitalIn=s[22], digitalOut=s[23], padding=None)

	return up

## 16/16

In [4]:
Path_non_decoded_files = "/Users/fpbattaglia/Dropbox/Data/Totalsync/477116"
Path_decoded = os.path.join(Path_non_decoded_files, 'decoded')

for filename in os.listdir(Path_non_decoded_files):
    if filename.endswith(".b64"):
        bp=os.path.join(Path_non_decoded_files, filename)

        num_lines = count_lines(bp)
        log_duration = num_lines/1000/60
        print(bp)
        print(f'{num_lines} packets, ~{log_duration:0.2f} minutes')


        # Decode and create new dataset
        data = np.zeros(num_lines, dtype=np_DataPacketType_withDigital)
        print(len(data))
        non_digital_names = list(np_DataPacketType_noDigital.names)

        with open(bp, 'rb') as bf:
            for nline, line in enumerate(tqdm(bf, total=num_lines)):
                bl = cobs.decode(base64.b64decode(line[:-1])[:-1])
                dp = unpack_data_packet(bl)
                data[non_digital_names][nline] = np.frombuffer(bl[:-8], dtype=np_DataPacketType_noDigital)
                digital_arr = np.frombuffer(bl[-8:], dtype=np.uint8)
                data[nline]['digital_in'] = np.hstack([np.unpackbits(digital_arr[1]), np.unpackbits(digital_arr[0])])
                data[nline]['digital_out'] = np.hstack([np.unpackbits(digital_arr[3]), np.unpackbits(digital_arr[2])])

        #Check for packetID jumps
        jumps = np.unique(np.diff(data['packetID']))

       # assert(len(jumps) and jumps[0] == 1)
        data['digital_in']=np.flip(data['digital_in'], 1)
        data['digital_out']=np.flip(data['digital_out'], 1)
        decoded = {"analog":data['analog'], "digitalIn":data['digital_in'], "digitalOut":data['digital_out'], "startTS":data['us_start'], "transmitTS":data['us_end'], "longVar":data['states'], "packetNums":data['packetID']}
        name=os.path.splitext(filename)[0]
        print(name)
        path=os.path.join(Path_decoded, name +'_decoded'+'.mat')
        savemat(path, decoded)

/Users/fpbattaglia/Dropbox/Data/Totalsync/477116/20251118-152602_925.b64
1383156 packets, ~23.05 minutes
1383156


  0%|          | 0/1383156 [00:00<?, ?it/s]

20251118-152602_925


In [9]:
data[0]

np.void((0, 72, 15781, 10762, 14763993, 14764174, [1798,    0, 1703, 2218, 2403, 2218, 2405, 4052], [4783,  291,    0,    0,    0,    0,    0,  181], [0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]), dtype=[('type', 'u1'), ('size', 'u1'), ('crc16', '<u2'), ('packetID', '<u4'), ('us_start', '<u4'), ('us_end', '<u4'), ('analog', '<u2', (8,)), ('states', '<u4', (8,)), ('digital_in', '<u2', (16,)), ('digital_out', '<u2', (16,))])

In [10]:
digital_in = data['digital_in']